# Iris Classifier using Vertex AI


## Overview

In this tutorial, you build a scikit-learn model and deploy it on infer in local environment using Google Cloud Storage for logging and tracking model and data


### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

### Costs

This tutorial uses billable components of Google Cloud:

* Vertex AI
* Cloud Storage

Learn about [Vertex AI
pricing](https://cloud.google.com/vertex-ai/pricing), [Cloud Storage
pricing](https://cloud.google.com/storage/pricing), 

## Get started

### Install Vertex AI SDK for Python and other required packages



In [22]:
# Vertex SDK for Python
import sys

!{sys.executable} -m pip install --upgrade google-cloud-aiplatform scikit-learn

/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=2343) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Defaulting to user installation because normal site-packages is not writeable


### Set Google Cloud project information
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [23]:
PROJECT_ID = "mlops-week-1-499905"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [24]:
BUCKET_URI = f"gs://mlops-course-week1-unique"  # @param {type:"string"}
MODEL_ARTIFACT_DIR="iris_classifier/model"

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [25]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Creating gs://mlops-course-week1-unique/...
ServiceException: 409 A Cloud Storage bucket named 'mlops-course-week1-unique' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

In [26]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

## Simple Decision Tree model
Build a Decision Tree model on iris data

In [27]:
import os
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from google.cloud import storage
from datetime import datetime
from feast import FeatureStore
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

INPUT_BUCKET = "iris-training-data-bucket"
OUTPUT_BUCKET = "mlops-course-week1-unique"
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
storage_client = storage.Client()
mlflow.set_experiment("iris_experiment")

/tmp/ipykernel_2343/864334635.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


<Experiment: artifact_location='/home/athipse/21F3002319_MLOPS_WEEKLY_ASSIGNMENT/mlruns/1', creation_time=1784610047389, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784610047389, lifecycle_stage='active', name='iris_experiment', tags={}, trace_location=None, workspace='default'>

In [28]:
# Download training data
store = FeatureStore(repo_path="feature_repo")

entity_df = pd.read_parquet("feature_repo/data/iris_train.parquet")[
    ["iris_id", "event_timestamp"]
]

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],
).to_df()

In [29]:
labels = pd.read_parquet("feature_repo/data/iris.parquet")[["iris_id", "species"]]

training_df = training_df.merge(labels, on="iris_id", how="left")

In [30]:
X_train = training_df[
    [
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width",
    ]
]

y_train = training_df["species"]

In [31]:
eval_df = pd.read_parquet("feature_repo/data/iris_eval.parquet")[
    ["iris_id", "event_timestamp"]
]

eval_df = store.get_historical_features(
    entity_df=eval_df,
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],
).to_df()

eval_df = eval_df.merge(labels, on="iris_id", how="left")

X_eval = eval_df[
    [
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width",
    ]
]

y_eval = eval_df["species"]

In [32]:
depths = [2, 3, 4]
criterions = ["gini", "entropy"]

In [33]:
for depth in depths:
    for criterion in criterions:
        with mlflow.start_run():
            model = DecisionTreeClassifier(
                max_depth=depth,
                criterion=criterion,
                random_state=1
            )

            model.fit(X_train, y_train)

            preds = model.predict(X_eval)

            accuracy = accuracy_score(y_eval, preds)

            precision = precision_score(
                y_eval,
                preds,
                average="macro"
            )

            mlflow.log_param("max_depth", depth)
            mlflow.log_param("criterion", criterion)

            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)

            mlflow.sklearn.log_model(
                model,
                name="iris_model",
                registered_model_name="IrisClassifier"
            )

Registered model 'IrisClassifier' already exists. Creating a new version of this model...
Created version '7' of model 'IrisClassifier'.
Registered model 'IrisClassifier' already exists. Creating a new version of this model...
Created version '8' of model 'IrisClassifier'.
Registered model 'IrisClassifier' already exists. Creating a new version of this model...
Created version '9' of model 'IrisClassifier'.
Registered model 'IrisClassifier' already exists. Creating a new version of this model...
Created version '10' of model 'IrisClassifier'.
Registered model 'IrisClassifier' already exists. Creating a new version of this model...
Created version '11' of model 'IrisClassifier'.
Registered model 'IrisClassifier' already exists. Creating a new version of this model...
Created version '12' of model 'IrisClassifier'.


In [34]:
import pickle
import joblib

model_path = "artifacts/model.joblib"
os.makedirs("artifacts", exist_ok=True)
joblib.dump(model, model_path)

['artifacts/model.joblib']

### Upload model artifacts and custom code to Cloud Storage

Before you can deploy your model for serving, Vertex AI needs access to the following files in Cloud Storage:

* `model.joblib` (model artifact)
* `preprocessor.pkl` (model artifact)

Run the following commands to upload your files:

In [35]:
output_bucket = storage_client.bucket(OUTPUT_BUCKET)
output_bucket.blob(
    f"{timestamp}/model.joblib"
).upload_from_filename(model_path)